In [1]:
import os
import time
import pickle
import threading
import numpy as np # type: ignore
from datetime import datetime # type: ignore
import matplotlib.pyplot as plt # type: ignore
import matplotlib.animation as animation # type: ignore
from matplotlib.patches import Circle, FancyArrowPatch # type: ignore
from typing import Dict, Any


### User-Defined

In [2]:
# POTENTIAL FORCES
# =============================================================================
from pot_forces import force_attractive, vforce_attractive, vforce_repulsive, mforce_repulsive

def create_vectorized_force_field(ax, x_lim, y_lim, resolution, decel_point, obs_buff):
    """
    Create a quiver plot and an update function that computes forces vectorized.
    Returns (quiver, update_func) where update_func(frame, goal, obs_pos, obs_r,
                                                    gain_attract=1.0, gain_repuls=3.0)
    """
    # Pre‑compute the grid once (contiguous in memory)
    x_grid = np.linspace(x_lim[0], x_lim[1], resolution)
    y_grid = np.linspace(y_lim[0], y_lim[1], resolution)
    X, Y = np.meshgrid(x_grid, y_grid)
    pos = np.column_stack((X.ravel(), Y.ravel()))          # shape (N_grid, 2)

    # Initialise empty quiver
    U = np.zeros_like(X)
    V = np.zeros_like(Y)
    quiver = ax.quiver(X, Y, U, V, alpha=0.6, scale=20, width=0.005)

    # Pre‑allocate force arrays (will be overwritten)
    F_total = np.empty_like(pos)

    def update_force(frame, goal, obs_pos, obs_r,
                     gain_attract=1.0, gain_repuls=3.0):
        """
        goal     : (2,) array – current goal position
        obs_pos  : (M,2) array – obstacle positions (may be empty)
        obs_r    : (M,)  array – obstacle radii
        """
        # --- Attractive force ---
        F_attr = vforce_attractive(pos, goal, decel_point, gain_attract)

        # --- Repulsive force ---
        if len(obs_pos) > 0:
            r_buff_arr = np.full_like(obs_r, obs_buff)
            F_rep = mforce_repulsive(pos, obs_pos, obs_r, r_buff_arr, gain_repuls)
            F_total[:] = F_attr + F_rep
        else:
            F_total[:] = F_attr

        # Reshape to grid for quiver
        U_new = F_total[:, 0].reshape(X.shape)
        V_new = F_total[:, 1].reshape(Y.shape)
        quiver.set_UVC(U_new, V_new)
        return quiver

    return quiver, update_force


In [3]:
# HIGH-LEVEL ANIMATION (optimized)
# =============================================================================
from plot_elements import setup_axes, auto_limits
from plot_elements import add_robot_circle, add_orientation_arrow, add_trail
from plot_elements import add_goal_marker, add_text_panel

def animate_environment(ax, params, log_data,
                        x_lim=None, y_lim=None, show_forces=False,
                        resolution=15, decel_point=100, obs_buff=100):
    # Unpack
    t = log_data['t']
    pos = log_data['pos']
    theta = log_data['theta']
    throttle = log_data['throttle']
    steering = log_data['steering']
    goal = log_data['goal']
    pos_obs_seq = log_data['pos_obs']
    r_obs_seq = log_data['r_obs']

    n_frames = len(pos)
    robot_radius = params.radius

    if x_lim is None or y_lim is None:
        x_lim, y_lim = auto_limits(pos[:,0], pos[:,1], robot_radius)
    setup_axes(ax, x_lim, y_lim, title='Robot Motion')

    # Visual elements
    robot_patch = add_robot_circle(ax, pos[0,0], pos[0,1], robot_radius)
    arrow_line, arrow_head = add_orientation_arrow(ax, pos[0,0], pos[0,1], theta[0], robot_radius)
    trail = add_trail(ax, pos[0,0], pos[0,1])
    throttle_txt, steering_txt = add_text_panel(ax)
    time_title = ax.set_title('')
    goal_marker = add_goal_marker(ax, goal[0,0], goal[0,1])

    # Obstacle patches
    max_obstacles = max((len(obs) for obs in pos_obs_seq if obs is not None), default=0)
    obstacle_patches = []
    for _ in range(max_obstacles):
        circ = Circle((0,0), 0, fc='red', ec='black', alpha=0.6, lw=1.5, zorder=2, visible=False)
        ax.add_patch(circ)
        obstacle_patches.append(circ)

    # Force field
    force_quiver = None
    update_force = None
    if show_forces:
        force_quiver, update_force = create_vectorized_force_field(
            ax, x_lim, y_lim, resolution, decel_point, obs_buff
        )

    # Timing
    t0 = t[0]
    rel_ts = t - t0
    dt_ms = np.diff(rel_ts)
    interval_ms = int(np.median(dt_ms)) if len(dt_ms) > 0 else 33

    def update_obstacles(frame):
        for patch in obstacle_patches:
            patch.set_visible(False)
        obs_pos = pos_obs_seq[frame]
        obs_r = r_obs_seq[frame]
        if obs_pos is None or len(obs_pos) == 0:
            return
        for i, (center, rad) in enumerate(zip(obs_pos, obs_r)):
            if i >= len(obstacle_patches):
                break
            obstacle_patches[i].center = (center[0], center[1])
            obstacle_patches[i].set_radius(rad)
            obstacle_patches[i].set_visible(True)

    update_obstacles(0)

    # Force cache
    last_goal = None
    last_obs_pos = None
    last_obs_r = None

    def update(frame):
        nonlocal last_goal, last_obs_pos, last_obs_r   # <-- FIX HERE

        if frame >= n_frames:
            return []

        # Robot
        robot_patch.center = (pos[frame,0], pos[frame,1])
        dx = robot_radius * np.cos(theta[frame])
        dy = robot_radius * np.sin(theta[frame])
        arrow_line.set_data([pos[frame,0], pos[frame,0] + dx],
                            [pos[frame,1], pos[frame,1] + dy])
        arrow_head.set_positions((pos[frame,0] + dx, pos[frame,1] + dy),
                                 (pos[frame,0] + dx*0.7, pos[frame,1] + dy*0.7))

        # Trail
        trail.set_data(pos[:frame+1,0], pos[:frame+1,1])

        # Text
        throttle_txt.set_text(f'Throttle: {throttle[frame]:.2f}')
        steering_txt.set_text(f'Steering: {steering[frame]:.2f}')
        time_title.set_text(f'Robot Motion At Time = {rel_ts[frame]/1000.0:.2f} sec')

        # Goal
        goal_marker.set_data([goal[frame,0]], [goal[frame,1]])

        # Obstacles
        update_obstacles(frame)

        # Force field update (only if changed)
        if show_forces and update_force is not None:
            curr_goal = goal[frame]
            curr_obs_pos = pos_obs_seq[frame]
            curr_obs_r = r_obs_seq[frame]

            force_changed = False
            if last_goal is None or not np.array_equal(curr_goal, last_goal):
                force_changed = True
            elif curr_obs_pos is None and last_obs_pos is not None:
                force_changed = True
            elif curr_obs_pos is not None:
                if last_obs_pos is None or len(curr_obs_pos) != len(last_obs_pos):
                    force_changed = True
                elif not np.array_equal(curr_obs_pos, last_obs_pos) or \
                     not np.array_equal(curr_obs_r, last_obs_r):
                    force_changed = True

            if force_changed:
                if curr_obs_pos is None:
                    curr_obs_pos = np.empty((0,2))
                    curr_obs_r = np.empty(0)
                update_force(frame, curr_goal, curr_obs_pos, curr_obs_r,
                             gain_attract=1.0, gain_repuls=3.0)
                last_goal = curr_goal.copy()
                last_obs_pos = curr_obs_pos.copy() if len(curr_obs_pos) else None
                last_obs_r = curr_obs_r.copy() if len(curr_obs_r) else None

        artists = [robot_patch, arrow_line, arrow_head, trail,
                   throttle_txt, steering_txt, time_title, goal_marker] + obstacle_patches
        if show_forces and force_quiver is not None:
            artists.append(force_quiver)
        return artists

    anim = animation.FuncAnimation(ax.figure, update, frames=n_frames,
                                   interval=interval_ms, blit=True, repeat=False)
    return anim

In [4]:
# GLOBAL PARAMETERS
# =============================================================================
class GlobalParameters:
    def __init__(self):
        self.pos = np.array([])               # List of np.array([x, y])
        self.theta: float = None              # rad
        self.radius = 100.0 
        self.goal = np.array([])
        self.goal_tol = 20.0
        self.decel_point = 100
        self.obstacles = {}         # Dict of {'name': [np.array([x, y]), radius] }
        self.safety_buffer = 1 * self.radius
        self.timestamp: float = None          # msec
    
    def reset_data(self):
        self.pos = np.array([])               # List of np.array([x, y])
        self.theta: float = None              # rad
        self.goal = np.array([])
        self.obstacles = {}         # Dict of {'name': [np.array([x, y]), radius] }
        self.timestamp: float = None          # msec

    def get_obstacles(self):
        pos_obs = np.array([])
        r_obs = np.array([])
        if self.obstacles.keys():
            pos_obs = np.vstack([value[0] for value in self.obstacles.values()])
            r_obs = np.vstack([value[1] for value in self.obstacles.values()])
        return pos_obs, r_obs

def save_data(data: Dict):
    os.makedirs("build", exist_ok=True)
    file_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")   # e.g. 2026-05-30_14-25-08
    file_path = f"build/log_{file_name}.pkl"

    with open(file_path, "wb") as f:
        pickle.dump(data, f)

def load_data(file_name:str):
    file_path = f"build/{file_name}"
    with open(file_path, "rb") as f:
        return pickle.load(f)

params = GlobalParameters()


In [5]:
# TCP CLIENT
# =============================================================================
from tcp_client import TCPBroadcaster, TCPReceiver # type: ignore

class BidirectionalClient:
    def __init__(self, host='localhost', incoming_port=9999, outgoing_port=9998):
        self.receiver = TCPReceiver(host, incoming_port)
        self.broadcaster = TCPBroadcaster(host, outgoing_port)
        self._data_lock = threading.Lock()
        self._latest_data = {}

    def connect_broadcaster(self):
        self.broadcaster.connect()

    def disconnect_broadcaster(self):
        self.broadcaster.close()

    def connect_receiver(self):
        self.receiver.connect()

    def disconnect_receiver(self):
        self.receiver.close()

    def start_receiving(self):
        self.receiver.start_receiving(self._handle_received_data)

    def stop_receiving(self):
        self.receiver.stop_receiving()

    def get_latest_state(self):
        with self._data_lock:
            return self._latest_data.copy()
    
    def send_control(self, throttle, steering):
        self.broadcaster.send({"type": "control_update", "throttle": throttle, "steering": steering})
    
    def demand_data(self, tag: str):
        self.broadcaster.send({"type": "demand_data", "tag": tag})

    def _handle_received_data(self, data):
        with self._data_lock:
            data_type = data.get('type')
            match data_type:
                case "pos_update":
                    params.pos = np.array( [data['x'], data['y']] )
                    params.theta = data['theta']
                    params.timestamp = data['timestamp']

                case 'goal_update':
                    params.goal = np.array( [data['x'], data['y']] )

                case 'obstacle_update':
                    params.obstacles[data['name']] = [ np.array( [data['x'], data['y']] ), data['r'] ]

                case _:
                    self._latest_data = data


In [6]:
# CONTROL LOOP (logs data)
# =============================================================================
def run_control_loop(goal_gain=1.0, obs_gain=3.0, obs_buff=100.0):
    """Run reactive potential‑field controller and return logged data."""
    client = BidirectionalClient('localhost', 9999, 9998)
    client.connect_broadcaster()
    client.connect_receiver()
    client.start_receiving()

    log_t, log_pos, log_theta = [], [], []
    log_goal, log_pos_obs, log_r_obs = [], [], []
    log_throttle, log_steering = [], []

    idle_state = True
    params.reset_data()

    print("Waiting for state...")
    while True:
        if params.pos.any():
            break
        time.sleep(0.02)
    print("Orientation recieved. (1/3)")

    client.demand_data('goal_update')
    while True:
        if params.goal.any():
            break
        time.sleep(0.02)
    print("Goal recieved. (2/3)")
    
    client.demand_data('obstacle_update')
    while True:
        if params.obstacles.keys():
            break
        time.sleep(0.02)
    print("Constraints recieved. (3/3)")

    print("Control loop started.")
    try:
        while True:
            pos = params.pos
            theta = params.theta
            timestamp = params.timestamp
            goal = params.goal
            pos_obs, r_obs = params.get_obstacles()

            # Logging
            log_t.append(timestamp)
            log_pos.append(pos)
            log_theta.append(theta)
            log_goal.append(goal)
            log_pos_obs.append(pos_obs)
            log_r_obs.append(r_obs)

            # Force computation
            F_att = force_attractive(pos, goal, params.decel_point, goal_gain)
            F_tot = F_att

            if pos_obs.any():
                r_inner = r_obs + params.radius
                F_rep = vforce_repulsive(pos, pos_obs, r_inner, obs_buff, obs_gain)
                F_tot += F_rep.sum(axis=0)  

            theta_des = np.arctan2(F_tot[1], F_tot[0])
            theta_error = np.arctan2(np.sin(theta_des - theta), np.cos(theta_des - theta))
            steering = np.clip(theta_error, -1.0, 1.0)

            force_mag = np.linalg.norm(F_tot)
            throttle = force_mag * (1.0 - 0.5 * abs(steering)) # Throttle reduction when turning


            # Final clamping (though already safe)
            throttle = np.clip(throttle, -1.0, 1.0)
            steering = np.clip(steering, -1.0, 1.0)

            error = pos - goal
            if np.hypot(error[0], error[1]) < params.goal_tol:
                throttle, steering = 0.0, 0.0
                if not idle_state:
                    print("Goal reached – stopping.")
                    idle_state = True
            else:
                idle_state = False

            log_throttle.append(throttle)
            log_steering.append(steering)
            client.send_control(throttle, steering)
            time.sleep(0.0166667)

    except KeyboardInterrupt:
        print("\nInterrupted by user.")
    finally:
        client.send_control(0.0, 0.0)
        client.stop_receiving()
        print("Control loop terminated.")

    return {
        't': np.array(log_t),               # shape (N,)
        'pos': np.array(log_pos),           # shape (N,2)
        'theta': np.array(log_theta),       # shape (N,)
        'goal': np.array(log_goal),         # shape (N,2)
        'pos_obs': np.array(log_pos_obs),   # N list of (M_i,2) arrays
        'r_obs': np.array(log_r_obs),       # N list of (M_i,) arrays
        'throttle': np.array(log_throttle), # shape (N,)
        'steering': np.array(log_steering), # shape (N,)
    }


# Main

In [19]:
# Run the control loop to generate log data
params.goal_tol = 40
params.decel_point = 200
params.safety_buffer = 500
log_data = run_control_loop(goal_gain = 1.0, obs_gain = 0.3,
                            obs_buff=params.safety_buffer)
save_data(log_data)

🎧 TCPReceiver initialized for port 9999
📡 TCPBroadcaster initialized for port 9998
✅ Connected to localhost:9998
✅ Connected to localhost:9999
Waiting for state...
Orientation recieved. (1/3)
Goal recieved. (2/3)
Constraints recieved. (3/3)
Control loop started.
Goal reached – stopping.
Goal reached – stopping.
Goal reached – stopping.

Interrupted by user.
Receive loop ended.
Control loop terminated.


In [7]:
log_data = load_data("log_2026-05-30_15-08-29.pkl")

In [8]:
# Compute axis limits from logged robot positions
robot_radius = params.radius
x_lim, y_lim = auto_limits(log_data['pos'][:, 0], log_data['pos'][:, 1], robot_radius)

# Create figure and animation
fig, ax = plt.subplots(figsize=(10, 8))
anim = animate_environment(
    ax, params, log_data,
    show_forces=True,
    resolution=15,
    decel_point=params.decel_point,
    obs_buff=params.safety_buffer
)
    
    
# Export to MP4 (as before)
output_dir = "build"
file_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")   # e.g. 2026-05-30_14-25-08
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, file_name+'.mp4')
total_seconds = (log_data['t'][-1] - log_data['t'][0]) / 1000.0
n_frames = len(log_data['t'])
fps = n_frames / total_seconds if total_seconds > 0 else 30
writer = animation.FFMpegWriter(fps=fps, bitrate=1800)
anim.save(output_path, writer=writer)
plt.close(fig)
print(f"Animation saved to {output_path} (fps = {fps:.0f})")

Animation saved to build\robot_animation.mp4 (fps = 53)
